# Assignment 1: Texture Classification with MLPs and CNNs (DTD)

In this assignment, you will explore how deep learning models handle visual recognition tasks by building classifiers on the [Describable Textures Dataset (DTD)](https://www.robots.ox.ac.uk/~vgg/data/dtd/). 

We begin with a simple multilayer perceptron (MLP) applied to raw pixels, in this process you may summarize the limitations of MLP in handling image data. Then, we will build and train a convolutional neural network (CNN) from scratch, learning how convolutional layers capture spatial patterns more effectively. Through evaluation and analysis, we will compare the performance of MLPs and CNNs, and reflect on what makes CNNs especially suitable for image understanding.

**Submission instructions**: 
- Complete codes in sections below, then rename this file as `<studentId>_<fullName>_assignment1.ipynb` and submit it to Moodle (an example of file name: `16483715_ShichaoMA_assignment1.ipynb`).
- Please keep all Outputs shown on this jupyter notebook (do not clear output after you finish the assignment).

**Also fill in the followings**:
- Name:WenjunYu
- Student ID:25480677

<hr>

DO NOT MODIFY THIS SECTION

In [47]:
import random
import time

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torchvision.transforms as T
from torchvision.datasets import DTD
import torch.nn.functional as F

In [48]:
# For reproducibility, DO NOT MODIFY.
seed = 42
random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
# For determinism (can slow down a bit)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [49]:
# Use CUDA if GPU is available, and use CPU otherwise.
if torch.cuda.is_available():
    device = 'cuda'
else:
    device = 'cpu'

print(f"Using device: {device}")

Using device: cuda


<hr>

## Part A: Data Pipeline (15%)
**Task: Constructing dataloaders for DTD using PyTorch.**

- You may use the dataset class provided by Torchvision: https://docs.pytorch.org/vision/main/generated/torchvision.datasets.DTD.html#torchvision.datasets.DTD
- Complete the following function `get_dataloaders` to build data loaders for the training set, the validation set, and the test set.
- Apply the following data processing:
    - Resize images to a fixed size of 64×64.
    - Convert to tensors and normalize pixel values using `mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]`.
    - Apply at least one data augmentation technique (e.g., random crop, horizontal flip, or color jitter).
    - Wrap everything in PyTorch DataLoaders.

In [50]:
def get_dataloaders(batch_size):
    import torch
    import torchvision.transforms as T
    from torchvision.datasets import DTD
    from torch.utils.data import DataLoader
    
    mean = [0.485, 0.456, 0.406]
    std = [0.229, 0.224, 0.225]
    
    # stronger train-time augmentation
    train_tfms = T.Compose([
        T.RandomResizedCrop(64, scale=(0.6, 1.0), ratio=(0.75, 1.33)),
        T.RandomHorizontalFlip(p=0.5),
        T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
        T.RandomGrayscale(p=0.05),
        T.ToTensor(),
        T.Normalize(mean=mean, std=std)
    ])
    eval_tfms = T.Compose([
        T.Resize((64, 64)),
        T.ToTensor(),
        T.Normalize(mean=mean, std=std)
    ])
    
    root = "./data"
    train_set = DTD(root=root, split="train", transform=train_tfms, download=False)
    val_set   = DTD(root=root, split="val",   transform=eval_tfms,   download=False)
    test_set  = DTD(root=root, split="test",  transform=eval_tfms,   download=False)
    
    kwargs = {
        "batch_size": batch_size,
        "num_workers": 8,
        "persistent_workers": True
    }
    if torch.cuda.is_available():
        kwargs.update({"pin_memory": True})
    
    train_loader = DataLoader(train_set, shuffle=True,  **kwargs)
    val_loader   = DataLoader(val_set,   shuffle=False, **kwargs)
    test_loader  = DataLoader(test_set,  shuffle=False, **kwargs)
    
    return train_loader, val_loader, test_loader


In [51]:
# Construct dataloaders, you can adjust batch_size according to your GPU.
train_loader, val_loader, test_loader = get_dataloaders(batch_size=128)

<hr>

# Part B: MLP Implementation and Model Training (40%)

**Task: Build and train an MLP on flattened DTD images.**
- Complete the class `MLP` to implement an MLP model with two hidden layers (hidden size 512 and 256).
- Choose suitable loss function and optimization algorithm to train the model for at least 30 epochs.
- Choose one suitable evaluation metric for this task.
- During training, monitor the evaluation metric computed over the *validation set*.
- After training, compute the evaluation metric over the *test set*.
- *Bonus (optional)*: apply dropout regularization, [lr scheduler](https://docs.pytorch.org/docs/stable/generated/torch.optim.lr_scheduler.LRScheduler.html), and early stopping.

**NB: There is no requirements for the accuracy level for the MLP model.**

In [52]:
class MLP(nn.Module):
    # Feel free to add more augments if needed.
    def __init__(self, in_dim, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_dim, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),  # increased dropout
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),  # increased dropout
            nn.Linear(256, num_classes)
        )

    def forward(self, x):  # x is the input image
        return self.net(x)

In [53]:
# create a model instance
mlp_model = MLP(in_dim=3*64*64, num_classes=47).to(device)

In [ ]:
# Complete this function for model training, evaluation, and testing.
# This function will also be used for training CNN later.
# Feel free to add more augments if needed.
def train_evaluate_and_test_model(model, lr, max_epochs, train_loader, val_loader, test_loader):
    from copy import deepcopy
    device_ = device  # use global device set earlier
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)  # label smoothing
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)  # weight decay
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)

    def accuracy(logits, targets):
        preds = logits.argmax(dim=1)
        return (preds == targets).float().mean().item()

    best_val_acc = -1.0
    best_state = None
    patience = 100
    epochs_no_improve = 0

    for epoch in range(1, max_epochs + 1):
        # Train
        model.train()
        train_loss_sum, train_correct, train_count = 0.0, 0, 0
        for imgs, labels in train_loader:
            imgs = imgs.to(device_)
            labels = labels.to(device_)
            optimizer.zero_grad()
            logits = model(imgs)
            loss = criterion(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # light grad clip
            optimizer.step()
            
            train_loss_sum += loss.item() * labels.size(0)
            train_correct += (logits.argmax(dim=1) == labels).sum().item()
            train_count += labels.size(0)
        train_loss = train_loss_sum / max(1, train_count)
        train_acc = train_correct / max(1, train_count)

        # Validate
        model.eval()
        val_loss_sum, val_correct, val_count = 0.0, 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs = imgs.to(device_)
                labels = labels.to(device_)
                logits = model(imgs)
                loss = criterion(logits, labels)
                val_loss_sum += loss.item() * labels.size(0)
                val_correct += (logits.argmax(dim=1) == labels).sum().item()
                val_count += labels.size(0)
        val_loss = val_loss_sum / max(1, val_count)
        val_acc = val_correct / max(1, val_count)

        scheduler.step(val_acc)

        print(f"Epoch {epoch:03d}/{max_epochs} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print("Early stopping triggered.")
                break

    # Load best weights
    if best_state is not None:
        model.load_state_dict(best_state)

    # Test
    model.eval()
    test_loss_sum, test_correct, test_count = 0.0, 0, 0
    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs = imgs.to(device_)
            labels = labels.to(device_)
            logits = model(imgs)
            loss = criterion(logits, labels)
            test_loss_sum += loss.item() * labels.size(0)
            test_correct += (logits.argmax(dim=1) == labels).sum().item()
            test_count += labels.size(0)
    test_loss = test_loss_sum / max(1, test_count)
    test_acc = test_correct / max(1, test_count)
    print(f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}")

    # after training, return the trained model.
    return model

In [55]:
# Call the function to train, evaluate, and test the MLP model.
mlp_model = train_evaluate_and_test_model(mlp_model, 
                                          lr=1e-3, 
                                          max_epochs=100, 
                                          train_loader=train_loader, 
                                          val_loader=val_loader, 
                                          test_loader=test_loader)

Epoch 001/100 | Train Loss: 5.2107 Acc: 0.0245 | Val Loss: 3.8492 Acc: 0.0261
Epoch 002/100 | Train Loss: 3.9965 Acc: 0.0245 | Val Loss: 3.8479 Acc: 0.0250
Epoch 002/100 | Train Loss: 3.9965 Acc: 0.0245 | Val Loss: 3.8479 Acc: 0.0250
Epoch 003/100 | Train Loss: 3.8898 Acc: 0.0282 | Val Loss: 3.8498 Acc: 0.0255
Epoch 003/100 | Train Loss: 3.8898 Acc: 0.0282 | Val Loss: 3.8498 Acc: 0.0255
Epoch 004/100 | Train Loss: 3.8816 Acc: 0.0261 | Val Loss: 3.8505 Acc: 0.0223
Epoch 004/100 | Train Loss: 3.8816 Acc: 0.0261 | Val Loss: 3.8505 Acc: 0.0223
Epoch 005/100 | Train Loss: 3.8799 Acc: 0.0250 | Val Loss: 3.8479 Acc: 0.0271
Epoch 005/100 | Train Loss: 3.8799 Acc: 0.0250 | Val Loss: 3.8479 Acc: 0.0271
Epoch 006/100 | Train Loss: 3.8569 Acc: 0.0245 | Val Loss: 3.8482 Acc: 0.0229
Epoch 006/100 | Train Loss: 3.8569 Acc: 0.0245 | Val Loss: 3.8482 Acc: 0.0229
Epoch 007/100 | Train Loss: 3.8801 Acc: 0.0234 | Val Loss: 3.8429 Acc: 0.0202
Epoch 007/100 | Train Loss: 3.8801 Acc: 0.0234 | Val Loss: 3.842

<hr>

## Part C: CNN Implementation (25%)

**Task: Build a CNN on DTD images.**
- Complete the class `smallCNN` to implement a small CNN model with the following layers:
    - Conv with 32 kernels (size: 3x3; padding: 1)
    - MaxPooling with kernel size 2
    - Conv with 32 kernels (size: 3x3; padding: 1)
    - MaxPooling with kernel size 2
    - Conv with 32 kernels (size: 3x3; padding: 1)
    - MaxPooling with kernel size 2
    - Then flatten the feature map and apply FC layers for classification.
    
- Specify proper parameters for the convolutional layers such that the dimensions could match.
- Use proper nonlinear activations in correct places.
    
- Call the function `train_evaluate_and_test_model` previously defined to train the model for at least 30 epochs.
- *Bonus (optional)*: apply [BatchNorm](https://d2l.ai/chapter_convolutional-modern/batch-norm.html) in proper places.

**NB: There is no requirements for the accuracy level for the MLP model.**

In [56]:
class SmallCNN(nn.Module):
    # Feel free to add more augments if needed.
    # For simplicity, you can just hard code the CNN architecture specified above.
    def __init__(self, num_classes=47):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2),  # 64 -> 32
            nn.Dropout2d(p=0.05),
            
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2),  # 32 -> 16
            nn.Dropout2d(p=0.05),
            
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2)   # 16 -> 8
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 8 * 8, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),  # stronger dropout
            nn.Linear(128, num_classes)
        )
    
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [57]:
# create an instance of the model and do training.
cnn_model = SmallCNN().to(device)
cnn_model = train_evaluate_and_test_model(cnn_model, 
                                          lr=1e-3, 
                                          max_epochs=100, 
                                          train_loader=train_loader, 
                                          val_loader=val_loader, 
                                          test_loader=test_loader)

Epoch 001/100 | Train Loss: 3.8889 Acc: 0.0255 | Val Loss: 3.8351 Acc: 0.0394
Epoch 002/100 | Train Loss: 3.8306 Acc: 0.0351 | Val Loss: 3.8195 Acc: 0.0420
Epoch 002/100 | Train Loss: 3.8306 Acc: 0.0351 | Val Loss: 3.8195 Acc: 0.0420
Epoch 003/100 | Train Loss: 3.8190 Acc: 0.0314 | Val Loss: 3.7937 Acc: 0.0378
Epoch 003/100 | Train Loss: 3.8190 Acc: 0.0314 | Val Loss: 3.7937 Acc: 0.0378
Epoch 004/100 | Train Loss: 3.8075 Acc: 0.0415 | Val Loss: 3.7798 Acc: 0.0521
Epoch 004/100 | Train Loss: 3.8075 Acc: 0.0415 | Val Loss: 3.7798 Acc: 0.0521
Epoch 005/100 | Train Loss: 3.7868 Acc: 0.0436 | Val Loss: 3.7551 Acc: 0.0676
Epoch 005/100 | Train Loss: 3.7868 Acc: 0.0436 | Val Loss: 3.7551 Acc: 0.0676
Epoch 006/100 | Train Loss: 3.7739 Acc: 0.0505 | Val Loss: 3.7306 Acc: 0.0707
Epoch 006/100 | Train Loss: 3.7739 Acc: 0.0505 | Val Loss: 3.7306 Acc: 0.0707
Epoch 007/100 | Train Loss: 3.7619 Acc: 0.0548 | Val Loss: 3.7196 Acc: 0.0622
Epoch 007/100 | Train Loss: 3.7619 Acc: 0.0548 | Val Loss: 3.719

## Part D: Comparisons and Reflections (20%)

- Compare the two models (e.g., model size, training speed, accuracy, overfitting,...)
    - For model size, you can count trainable parameters by `trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)`.
    - For training speed, you can use `start=time.time(); ...(model training codes)...; end=time.time();` to measure the running time.
- Summarize your observations throughtout the entire assignment, analyze your observations using the content you learned from the lectures.
- For this self-reflection, please refrain from using LLMs, but instead write down your own thoughts.

Write in this markdown cell. Add additional cells as you need.

In [59]:
# 对比：参数量与在 train/val/test 上的准确率与过拟合差值

def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

def evaluate_accuracy(model, loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device); y = y.to(device)
            logits = model(x)
            pred = logits.argmax(1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    return correct / max(1, total)

# 为训练集构建无数据增强的评估 loader，避免随机增强影响训练集准确率估计
mean = [0.485, 0.456, 0.406]; std = [0.229, 0.224, 0.225]
eval_tfms = T.Compose([T.Resize((64, 64)), T.ToTensor(), T.Normalize(mean, std)])
train_eval_set = DTD(root="./data", split="train", transform=eval_tfms, download=False)
loader_kwargs = {"batch_size": 256, "num_workers": 4}
if torch.cuda.is_available():
    loader_kwargs.update({"pin_memory": True})
train_eval_loader = DataLoader(train_eval_set, shuffle=False, **loader_kwargs)

# 统计参数量
mlp_params = count_params(mlp_model)
cnn_params = count_params(cnn_model)

# 评估各集上的准确率
mlp_train_acc = evaluate_accuracy(mlp_model, train_eval_loader, device)
mlp_val_acc   = evaluate_accuracy(mlp_model, val_loader, device)
mlp_test_acc  = evaluate_accuracy(mlp_model, test_loader, device)

cnn_train_acc = evaluate_accuracy(cnn_model, train_eval_loader, device)
cnn_val_acc   = evaluate_accuracy(cnn_model, val_loader, device)
cnn_test_acc  = evaluate_accuracy(cnn_model, test_loader, device)

print("Model Size (trainable params):")
print(f"  MLP : {mlp_params}")
print(f"  CNN : {cnn_params}\n")

print("Accuracy (train / val / test):")
print(f"  MLP : {mlp_train_acc:.4f} / {mlp_val_acc:.4f} / {mlp_test_acc:.4f}")
print(f"  CNN : {cnn_train_acc:.4f} / {cnn_val_acc:.4f} / {cnn_test_acc:.4f}\n")

print("Overfitting Gap (train - val, train - test):")
print(f"  MLP : {(mlp_train_acc-mlp_val_acc):.4f} , {(mlp_train_acc-mlp_test_acc):.4f}")
print(f"  CNN : {(cnn_train_acc-cnn_val_acc):.4f} , {(cnn_train_acc-cnn_test_acc):.4f}")

Model Size (trainable params):
  MLP : 6435375
  CNN : 287919

Accuracy (train / val / test):
  MLP : 0.0452 / 0.0356 / 0.0372
  CNN : 0.2431 / 0.1766 / 0.1750

Overfitting Gap (train - val, train - test):
  MLP : 0.0096 , 0.0080
  CNN : 0.0665 , 0.0681


## Comparison Summary

### Model Scale (Number of Parameters)
- **MLP:** 6,435,375  
- **CNN:** 287,919 (≈22.4× smaller)  

### Accuracy (train / val / test)
- **MLP:** 0.0452 / 0.0356 / 0.0372  
- **CNN:** 0.2431 / 0.1766 / 0.1750  

### Overfitting Analysis (train-val, train-test)
- **MLP:** 0.0096, 0.0080 → Essentially no overfitting; more like underfitting / learning failure (random guess for 47 classes ≈ 0.021)  
- **CNN:** 0.0665, 0.0681 → Some overfitting exists, but overall performance is significantly better than MLP  

### Key Conclusions
- CNN’s inductive biases (local connectivity, weight sharing, translation invariance) align better with image data, leading to substantial performance gains.  
- MLP loses spatial structure after flattening, making it difficult to learn texture patterns.  
- The current MLP is near random-level, while CNN has learned meaningful features but still shows some overfitting and potential for improvement.  

### Improvement Suggestions (from easiest to hardest)
1. **Training adjustments:** Increase number of epochs, adopt cosine/OneCycle learning rate schedule, reduce initial LR.  
2. **Stronger regularization:** Apply more aggressive data augmentation (RandAugment / Mixup / CutMix), fine-tune weight decay, adjust Dropout ratio.  
3. **Model enhancement:** Widen CNN channels or leverage lightweight pretraining (e.g., ResNet18 / ConvNeXt-T transfer learning).  
4. **Data-level adjustments:** While maintaining 64×64 input, try RandomResizedCrop with wider scale, or increase input resolution to retain more texture details.


### Self-Reflection

- **Observation:** The MLP accuracy is near random, with almost no difference between training and validation; the CNN achieves significantly higher accuracy but shows a ~0.05 gap between train and val/test. This indicates that the MLP failed to learn meaningful representations, whereas the CNN has fit the training set and begun to overfit.  

- **Analysis:** Flattening in the MLP destroys spatial relationships. Despite having a large number of parameters, it lacks inductive biases for locality and translation invariance. CNN’s local convolutions and weight sharing provide appropriate inductive biases, enabling the extraction of texture patterns from 64×64 images. The relatively low input resolution may limit the separability of complex textures, and current regularization and augmentation strategies are still insufficient to fully prevent CNN overfitting.  

- **Takeaways:** The advantage of convolutional structures in texture classification is evident. Simply increasing MLP capacity cannot replace proper structural inductive biases. Monitoring the validation set and applying early stopping are effective in controlling overfitting.
